In [2]:
# import the necessary packages
import numpy as np
import matplotlib.pyplot as plt
import os
import argparse
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import AveragePooling2D, Dropout, Dense, Flatten, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import classification_report
from imutils import paths

In [3]:
# construct the argument  parser and parse the argument

ap = argparse.ArgumentParser()
ap.add_argument("-d", "--dataset", required = True, help = "path to input dataset")
ap.add_argument("-p", "--plot", type = str, default = "plot.png", help = " path to output loss/accuracy plot")
ap.add_argument("-m", "--model", type = str, default = " mask_detectorch.model", help = "path to output face mask detector model")
args = vars(ap.parse_args())

usage: ipykernel_launcher.py [-h] -d DATASET [-p PLOT] [-m MODEL]
ipykernel_launcher.py: error: the following arguments are required: -d/--dataset


SystemExit: 2

C:\Anaconda3\envs\aiml\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
# initialize the initial learning rate, number of epochs to train and batch size
INIT_LR = 1e-4
EPOCHS = 20
BS = 32

In [5]:
# grab the list of image in our dataset directory, then initialize the list of data and class image 

print("[INFO] loading images...")
dataset_path = r"D:\Hope\AI couse tamil\7. Deep Learning\Face-Mask-Detection\dataset"
imagePaths = list(paths.list_images(dataset_path))
data = []
labels = []


[INFO] loading images...


In [6]:
len(imagePaths)

3846

In [7]:
imagePaths

['D:\\Hope\\AI couse tamil\\7. Deep Learning\\Face-Mask-Detection\\dataset\\without_mask\\0.jpg',
 'D:\\Hope\\AI couse tamil\\7. Deep Learning\\Face-Mask-Detection\\dataset\\without_mask\\0_0_aidai_0014.jpg',
 'D:\\Hope\\AI couse tamil\\7. Deep Learning\\Face-Mask-Detection\\dataset\\without_mask\\0_0_aidai_0029.jpg',
 'D:\\Hope\\AI couse tamil\\7. Deep Learning\\Face-Mask-Detection\\dataset\\without_mask\\0_0_aidai_0043.jpg',
 'D:\\Hope\\AI couse tamil\\7. Deep Learning\\Face-Mask-Detection\\dataset\\without_mask\\0_0_aidai_0074.jpg',
 'D:\\Hope\\AI couse tamil\\7. Deep Learning\\Face-Mask-Detection\\dataset\\without_mask\\0_0_aidai_0084.jpg',
 'D:\\Hope\\AI couse tamil\\7. Deep Learning\\Face-Mask-Detection\\dataset\\without_mask\\0_0_aidai_0136.jpg',
 'D:\\Hope\\AI couse tamil\\7. Deep Learning\\Face-Mask-Detection\\dataset\\without_mask\\0_0_anhu_0004.jpg',
 'D:\\Hope\\AI couse tamil\\7. Deep Learning\\Face-Mask-Detection\\dataset\\without_mask\\0_0_anhu_0020.jpg',
 'D:\\Hope\\AI c

In [8]:
"""
for imagePath in imagePaths:
    label = imagePath.split(os.path.sep)
    label = imagePath.split(os.path.sep)[-2]
    print(label)
"""

'\nfor imagePath in imagePaths:\n    label = imagePath.split(os.path.sep)\n    label = imagePath.split(os.path.sep)[-2]\n    print(label)\n'

In [9]:
"""
single_img = imagePaths[1].split(os.path.sep)[-2]
print(single_img)
image = load_img(imagePaths[1], target_size = (224,224))
print(image)
image = img_to_array(image)
print(image)
print(image.shape)
image = preprocess_input(image)
print(image)
print(image.shape)
len(image)
data.append(image)
print(len(data))
"""

'\nsingle_img = imagePaths[1].split(os.path.sep)[-2]\nprint(single_img)\nimage = load_img(imagePaths[1], target_size = (224,224))\nprint(image)\nimage = img_to_array(image)\nprint(image)\nprint(image.shape)\nimage = preprocess_input(image)\nprint(image)\nprint(image.shape)\nlen(image)\ndata.append(image)\nprint(len(data))\n'

In [10]:
# loop over the image path

for imagePath in imagePaths:
    # extract the class label from the filename 
    label = imagePath.split(os.path.sep)[-2]

    # load the input image (224*224) and preprocess it
    image = load_img(imagePath, target_size = (224, 224))
    #print((image))
    image = img_to_array(image)
    #print(len(image))
    #print(image)
    #print(image.shape)
    image = preprocess_input(image)
    #print(len(image))
    #print(image)    

    ## update the data and labels list, respectively
    data.append(image)
    labels.append(label)
    
    
    

C:\Anaconda3\envs\aiml\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [11]:
len(data)

3846

In [12]:
len(labels)

3846

In [13]:
# converts the data and labels to numpy array

data = np.array(data, dtype = "float32")
labels = np.array(labels)
#print(data.shape)

# perform one hot encoding on the labels

lb = LabelBinarizer()
labels = lb.fit_transform(labels)
labels = to_categorical(labels)

In [14]:
data.shape

(3846, 224, 224, 3)

In [15]:
labels

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [1., 0.],
       [1., 0.],
       [1., 0.]], shape=(3846, 2))

In [16]:
# parition the data into training and testing splits using 75% of the data for training and the remaining 25% for testing 

(trainX, testX, trainY, testY) = train_test_split(data, labels, test_size=0.20, stratify = labels, random_state = 42)

In [17]:
print(trainX.shape)
print(trainY.shape)
print(testX.shape)
print(testY.shape)

(3076, 224, 224, 3)
(3076, 2)
(770, 224, 224, 3)
(770, 2)


In [18]:
 # construct the training image generator for data agumentation:

aug = ImageDataGenerator(rotation_range = 20, zoom_range = 0.15,
                         width_shift_range = 0.2, height_shift_range = 0.2,
                         shear_range = 0.15, horizontal_flip = True, fill_mode = "nearest")

In [19]:
# load the mobilenetv2 network, ensuring the head FC layer sets are left off

baseModel = MobileNetV2(weights ="imagenet", include_top = False, 
                        input_tensor = Input(shape=(224, 224, 3)))

C:\Users\DELL\AppData\Local\Temp\ipykernel_15388\1864227953.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  baseModel = MobileNetV2(weights ="imagenet", include_top = False,


In [20]:
# construct the head of the model that will be placed on top of the base model

headModel = baseModel.output
headModel = AveragePooling2D(pool_size = (7, 7))(headModel)
headModel = Flatten(name="flatten")(headModel)
headModel = Dense(128, activation = "relu")(headModel)
headModel = Dropout(0.5)(headModel)
headModel = Dense(2, activation = "softmax")(headModel)

In [21]:
# place the head FC model on top of the model ( this will become the actual model will be train)

model = Model(inputs = baseModel.input, outputs = headModel)

In [22]:
# loop over all layers in the base model and freeze them so they will not be updated during the first training process

for layers in baseModel.layers:
    layers.trainable = False

In [23]:
# compile our model:

print("[INFO] compilling model...")
opt = Adam(lr=INIT_LR, decay=INIT_LR / EPOCHS)
#opt = Adam(learning_rate=INIT_LR)
model.compile(loss = "binary_crossentropy", optimizer=opt, metrics=["accuracy"])

[INFO] compilling model...


C:\Anaconda3\envs\aiml\Lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


ValueError: Argument(s) not recognized: {'lr': 0.0001}

In [ ]:
# train the head of the network:

print("[INFO] training head...")
H = model.fit (aug.flow(trainX, trainY, batch_size = BS),
               steps_per_epoch=len(trainX) //BS,
               validation_data =(testX, testY),
               validation_steps = len(testX) //BS,
               epochs = EPOCHS)


In [ ]:
# make prediction on the testing set:

print("[INFO] evaluating network...")
predIdxs = model.predict(testX, batch_size = BS)


In [ ]:
# for the image in the testing set we need find the index of the label with corresponding largest predicted probability:

predIdxs = np.argmax(predIdxs, axis = 1)

In [ ]:
# show a nicely formatted classification report:

print(classification_report(testY.argmax(axis=1), predIdxs, 
                            target_names = lb.classes_))

In [ ]:
# serialize the model disk:

print("[INFO] saving mask detector model...")
model.save(args["model_1"], save_format = "h5")

In [ ]:
# plot the training loss and accuracy:

N = EPOCHS 
plt.style.use("ggplot")
plt.figure()
plt.plot(np.arange(0, N), H.history["loss"], label ="train_loss")
plt.plot(np.arange(0, N), H.history["val_loss"], label = "val_loss")
plt.plot(np.arange(0, N), H.history["acc"], label = "train_acc")
plt.plot(np.arange(0, N), H.history["val_acc"], label = "val_acc")
plt.title("Training Loss and Accuracy")
plt.xlabel("Epoch #")
plt.ylabel("Loss/Accuracy")
plt.legend(loc = "lower left")
plt.savefig(arg["plot"])